# 1. Snowflake Setup and Export to Ossie (S3 Backbone)

This notebook builds the Snowflake side of the demo and exports the semantic view
to an Apache Ossie file on an S3 external stage. Databricks reads from the same
bucket automatically.

Order of work:
1. Set your database and schema.
2. Verify Iceberg tables and data on S3.
3. Create a semantic view over those tables.
4. Export the semantic view to Ossie YAML on the S3 stage.
5. (Optional) Create a suspended sync task for live demo.

Prerequisites:
- Storage integration `OSSIE_S3_INT` and external stage `OSSIE_S3_STAGE` exist
- External volume `OSSIE_ICEBERG_VOL` exists with Iceberg tables already loaded
- Role with ACCOUNTADMIN or sufficient privileges on the schema

## Step 1 - Set your database and schema

In [ ]:
DATABASE = "DEMOS"
SCHEMA   = "EXT_SEMANTIC_INTEROP"
print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Verify Iceberg tables on S3

The tables are Snowflake-managed Iceberg tables stored on S3. Both Snowflake and
Databricks read from the same physical Parquet files.

In [ ]:
-- Expected: EAST 750/5/12, WEST 700/5/11
SELECT c.region, SUM(o.order_amount) AS total_amount, COUNT(o.order_id) AS order_count, SUM(o.order_qty) AS total_qty
FROM {{DATABASE}}.{{SCHEMA}}.ORDERS o
JOIN {{DATABASE}}.{{SCHEMA}}.CUSTOMERS c USING (customer_id)
GROUP BY c.region ORDER BY c.region;

## Step 3 - Create the semantic view

The view defines two metrics (`TOTAL_ORDER_AMOUNT`, `ORDER_COUNT`), a region
dimension, and the ORDERS-to-CUSTOMERS relationship.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.SALES_SV
  TABLES (
    orders AS {{DATABASE}}.{{SCHEMA}}.ORDERS PRIMARY KEY (order_id),
    customers AS {{DATABASE}}.{{SCHEMA}}.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region,
    customers.customer_name AS customer_name
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount),
    orders.order_count AS COUNT(orders.order_id)
  )
  COMMENT = 'Sales star for Ossie interop demo (Iceberg on S3)';

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.SALES_SV
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;

## Step 4 - Export the semantic view to Ossie on S3

The COPY writes the Ossie YAML to `s3://snowflake-ossie-interop/ossie/ossie_from_snowflake.yaml`.
Databricks reads this file directly from the same bucket.

In [ ]:
COPY INTO @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE/ossie_from_snowflake.yaml
FROM (SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW('{{DATABASE}}.{{SCHEMA}}.SALES_SV'))
FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE
               ESCAPE_UNENCLOSED_FIELD = NONE COMPRESSION = NONE)
SINGLE = TRUE OVERWRITE = TRUE;

In [ ]:
LIST @{{DATABASE}}.{{SCHEMA}}.OSSIE_S3_STAGE;

## Step 5 - Sync Task (for live demo)

This task runs every 1 minute when resumed. It exports the latest semantic view to
S3 and imports the latest Databricks export. **Created suspended** -- enable during
demo, disable immediately after.

```sql
ALTER TASK DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_SYNC_TASK RESUME;
ALTER TASK DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_SYNC_TASK SUSPEND;
```

In [ ]:
CREATE OR REPLACE TASK {{DATABASE}}.{{SCHEMA}}.OSSIE_SYNC_TASK
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = '1 MINUTE'
  COMMENT = 'Exports semantic view to S3 and imports Databricks export. SUSPENDED by default.'
AS
BEGIN
  COPY INTO @DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_S3_STAGE/ossie_from_snowflake.yaml
  FROM (SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW('DEMOS.EXT_SEMANTIC_INTEROP.SALES_SV'))
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE
                 ESCAPE_UNENCLOSED_FIELD = NONE COMPRESSION = NONE)
  SINGLE = TRUE OVERWRITE = TRUE;

  LET yaml_content VARCHAR := (
    SELECT $1 FROM @DEMOS.EXT_SEMANTIC_INTEROP.OSSIE_S3_STAGE/ossie_from_databricks.yaml
    (FILE_FORMAT => 'DEMOS.EXT_SEMANTIC_INTEROP.RAW_TEXT_FMT')
  );
  IF (:yaml_content IS NOT NULL) THEN
    CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('DEMOS.EXT_SEMANTIC_INTEROP', :yaml_content);
  END IF;
END;

In [ ]:
SHOW TASKS IN SCHEMA {{DATABASE}}.{{SCHEMA}};

## Done

The Ossie file is at `s3://snowflake-ossie-interop/ossie/ossie_from_snowflake.yaml`.
Open notebook 2 in Databricks -- it reads directly from S3, no file transfer needed.